# 04 - Azure Unified Recommendations — Local Dev

Run and debug the **Azure unified recommendation job** locally.  
The 8,100-line Glue job is decomposed into testable pieces:

| Component | Glue dependency | Local replacement |
|---|---|---|
| `SimplifiedDatabaseConfig` | Spark JDBC | `LocalDatabaseConfig` (psycopg2) |
| `spark.sql(...)` on Iceberg | SparkSession | `LocalIcebergLoader` (Athena) |
| `GlueContext` / `Job` | awsglue | No-op stub |
| `GenericPricing` (Retail API) | aiohttp | Works as-is |
| `ThresholdRule.evaluate()` | None | Works as-is |
| `BaseAzureServiceRecommender` | None | Works as-is |
| `RightSizingEngine` | Spark (for init) | Athena-based init from notebook 02 |
| Iceberg writes | Spark | Skip — inspect results in-memory |

In [ ]:
import sys, os, json, logging
import pandas as pd

# Paths
nb_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.join(nb_dir, '..')
azure_glue_dir = os.path.join(project_root, 'src', 'glue', 'azure')
scripts_common = os.path.join(project_root, 'src', 's3', 'scripts', 'common')

sys.path.insert(0, nb_dir)
sys.path.insert(0, scripts_common)
# NOTE: We do NOT add azure_glue_dir to sys.path because the module has
#       top-level side effects (SparkContext, pip install, getResolvedOptions).
#       Instead we cherry-pick classes in the next cell.

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger('azure_dev')

print('Paths configured.')

## 1. Import Azure Classes (cherry-picked, no side effects)

We use `importlib` to load the module's source and extract only the
classes we need — avoiding the top-level `SparkContext()` / `getResolvedOptions()` calls.

In [ ]:
import importlib.util, types

def _load_azure_classes():
    """Load the azure module source and extract classes without executing main."""
    src_path = os.path.join(azure_glue_dir, 'azure_unified_recommendations.py')
    
    with open(src_path, 'r') as f:
        source = f.read()
    
    # Chop off everything after "# MAIN EXECUTION" to avoid side effects
    marker = '# MAIN EXECUTION'
    idx = source.find(marker)
    if idx > 0:
        source = source[:idx] + '\n# (main execution removed for local dev)\n'
    
    # Stub out problematic imports
    stub_preamble = '''
import sys as _sys

# --- Stubs for awsglue (not available locally) ---
class _StubModule:
    def __getattr__(self, name):
        return type('Stub', (), {'__init__': lambda *a,**k: None, '__getattr__': lambda s,n: None})

for _mod_name in ['awsglue', 'awsglue.context', 'awsglue.job', 'awsglue.utils']:
    if _mod_name not in _sys.modules:
        _sys.modules[_mod_name] = _StubModule()

# Stub getResolvedOptions
_sys.modules['awsglue.utils'].getResolvedOptions = lambda argv, params: {}
'''
    
    # Remove subprocess pip install lines
    import re
    source = re.sub(r'subprocess\.check_call\(\[sys\.executable.*?\]\)', 'pass', source)
    
    full_source = stub_preamble + source
    
    # Compile and exec into a module namespace
    mod = types.ModuleType('azure_unified')
    mod.__file__ = src_path
    exec(compile(full_source, src_path, 'exec'), mod.__dict__)
    return mod

azure_mod = _load_azure_classes()

# Extract the classes we need
ThresholdRule           = azure_mod.ThresholdRule
GenericPricing          = azure_mod.GenericPricing
AzureSkuNavigator       = azure_mod.AzureSkuNavigator
SavingsCalculator       = azure_mod.SavingsCalculator
BaseAzureServiceRecommender = azure_mod.BaseAzureServiceRecommender
AzureRecommendation     = azure_mod.AzureRecommendation
ConfigurationManager    = azure_mod.ConfigurationManager
create_service_recommender = azure_mod.create_service_recommender

print(f'Loaded {len([ThresholdRule, GenericPricing, SavingsCalculator, BaseAzureServiceRecommender])} core classes from azure_unified_recommendations.py')

## 2. Initialize Local Adapters

In [ ]:
from azure_local_adapter import (
    LocalDatabaseConfig,
    LocalIcebergLoader,
    build_local_args,
    aggregate_resource_metrics,
)

# --- Configuration ---
CLIENT_ID = os.getenv('CLIENT_ID', 'your-client-id')  # <-- set in .env
SUBSCRIPTION_ID = os.getenv('AZURE_SUBSCRIPTION_ID', '')  # optional
SERVICES = 'VM'  # comma-separated: 'VM,REDIS,COSMOSDB,AKS,DISK'

args = build_local_args(
    services=SERVICES,
    client_id=CLIENT_ID,
    subscription_id=SUBSCRIPTION_ID,
)
print('Job args:', json.dumps({k: v for k, v in args.items() if 'SECRET' not in k and 'PASSWORD' not in k}, indent=2))

In [ ]:
# Initialize local database config (replaces SimplifiedDatabaseConfig)
db_config = LocalDatabaseConfig(
    client_id=CLIENT_ID,
    subscription_id=SUBSCRIPTION_ID,
)
db_config.set_threshold_rule_class(ThresholdRule)

# Initialize Iceberg loader (replaces spark.sql)
iceberg = LocalIcebergLoader()

print('Local adapters ready.')

## 3. Load Service Configuration from RDS

In [ ]:
# Pick the first service to develop against
service_code = SERVICES.split(',')[0].strip()

service_config = db_config.get_service_config(service_code, cloud_provider='azure')
if service_config:
    print(f'Service config for "{service_code}":')
    for k, v in service_config.items():
        print(f'  {k:30s} = {v}')
else:
    print(f'No service config found for "{service_code}". Check the service_configuration table in RDS.')

## 4. Load Threshold Rules from RDS

In [ ]:
if service_config:
    service_name = service_config['service_name']
    rules = db_config.load_service_rules(service_name, cloud_provider='azure')
    
    print(f'Loaded {len(rules)} rules for "{service_name}":\n')
    for r in rules:
        print(f'  [{r.priority}] {r.rule_code:30s} severity={r.severity_level:8s} '
              f'conditions={len(r.evaluation_logic.get("conditions", []))}')
        for c in r.evaluation_logic.get('conditions', []):
            thresh = r.get_threshold_value(c['metric'])
            print(f'        {c["metric"]} {c.get("operator","?")} {thresh}')

## 5. Load Resources & Metrics from Iceberg (via Athena)

In [ ]:
if service_config:
    resources_table = service_config['resources_table_suffix']
    metrics_table = service_config['metrics_table_suffix']
    svc_filter = service_config['service_name_filter']
    
    print(f'Loading resources from: {resources_table}')
    print(f'Loading metrics from:   {metrics_table} (filter: {svc_filter})')
    
    resources_pdf = iceberg.load_resources(resources_table, CLIENT_ID)
    print(f'\nResources: {len(resources_pdf)} rows, columns: {list(resources_pdf.columns)}')
    display(resources_pdf.head(5))

In [ ]:
if service_config:
    metrics_pdf = iceberg.load_metrics(metrics_table, CLIENT_ID, svc_filter)
    print(f'Metrics: {len(metrics_pdf)} rows')
    
    if not metrics_pdf.empty:
        print(f'Distinct metric names:')
        display(metrics_pdf['metric_name'].value_counts())
        display(metrics_pdf.head(5))

## 6. Load Pricing Caches

In [ ]:
focus_table = args.get('FOCUS_TABLE_NAME', 'bronze_azure_focus_report_tbl')

print('Loading billing cost cache (Bronze FOCUS)...')
billing_cache = iceberg.load_billing_cost_cache(focus_table, CLIENT_ID)
print(f'  {len(billing_cache)} resources with billing data')

print('Loading SKU catalog cache...')
sku_cache = iceberg.load_sku_catalog_cache()
print(f'  {len(sku_cache)} SKU pricing entries')

## 7. Evaluate Rules for a Single Resource (step-by-step debug)

In [ ]:
if service_config and not resources_pdf.empty:
    # Pick the first resource
    sample_resource = resources_pdf.iloc[0].to_dict()
    resource_id = sample_resource.get('resource_id', '')
    
    print(f'=== Resource: {resource_id} ===')
    for k, v in sample_resource.items():
        if v is not None and str(v).strip():
            print(f'  {k:30s} = {v}')

In [ ]:
if service_config and not resources_pdf.empty:
    # Aggregate metrics for this resource
    agg_metrics = aggregate_resource_metrics(metrics_pdf, resource_id)
    
    print(f'Aggregated metrics for {resource_id}:')
    for k, v in sorted(agg_metrics.items()):
        print(f'  {k:40s} = {v:.4f}')

In [ ]:
if service_config and not resources_pdf.empty:
    # Evaluate each rule against this resource's metrics
    print(f'\n=== Rule Evaluation for {resource_id} ===\n')
    
    triggered_rules = []
    for rule in rules:
        result = rule.evaluate(agg_metrics, debug=True)
        status = 'TRIGGERED' if result else 'not triggered'
        print(f'  [{rule.rule_code:30s}] -> {status}')
        if result:
            triggered_rules.append(rule)
    
    print(f'\n{len(triggered_rules)} rule(s) triggered.')

In [ ]:
if service_config and triggered_rules:
    # Calculate savings for triggered rules
    for rule in triggered_rules:
        # Look up cost from billing cache
        annual_cost = billing_cache.get(resource_id.lower(), 0.0)
        if annual_cost == 0:
            # Fallback: try SKU catalog cache
            sku = sample_resource.get('vm_size') or sample_resource.get('sku_name') or ''
            region = sample_resource.get('region') or sample_resource.get('location') or ''
            annual_cost = sku_cache.get(
                (service_name.lower(), sku.lower(), region.lower()), 0.0
            )
        
        savings_pct = rule.get_savings_percentage() if hasattr(rule, 'get_savings_percentage') else 0
        savings = annual_cost * (savings_pct / 100) if savings_pct else 0
        
        print(f'Rule: {rule.rule_code}')
        print(f'  Annual cost:    ${annual_cost:,.2f}')
        print(f'  Savings %:      {savings_pct}%')
        print(f'  Est. savings:   ${savings:,.2f}')
        print(f'  Title:          {rule.title_template}')
        print(f'  Action:         {rule.recommended_actions}')
        print()

## 8. Batch: Process All Resources for a Service

Replicates the core loop of `_process_single_service()` using local adapters.

In [ ]:
if service_config and not resources_pdf.empty:
    batch_results = []
    
    for idx, row in resources_pdf.iterrows():
        rd = row.to_dict()
        rid = rd.get('resource_id', '')
        
        # Aggregate metrics
        m = aggregate_resource_metrics(metrics_pdf, rid)
        if not m:
            continue
        
        # Evaluate rules
        triggered = db_config.evaluate_all_rules(service_name, m)
        if not triggered:
            continue
        
        # Get cost
        annual_cost = billing_cache.get(rid.lower(), 0.0)
        if annual_cost == 0:
            sku = rd.get('vm_size') or rd.get('sku_name') or ''
            region = rd.get('region') or rd.get('location') or ''
            annual_cost = sku_cache.get(
                (service_name.lower(), sku.lower(), region.lower()), 0.0
            )
        
        for rule in triggered:
            savings_pct = rule.get_savings_percentage() if hasattr(rule, 'get_savings_percentage') else 0
            savings = annual_cost * (savings_pct / 100) if savings_pct else 0
            
            batch_results.append({
                'resource_id': rid,
                'resource_name': rd.get('resource_name') or rd.get('name') or rid,
                'rule_code': rule.rule_code,
                'severity': rule.severity_level,
                'annual_cost': annual_cost,
                'savings_pct': savings_pct,
                'estimated_savings': savings,
                'sku': rd.get('vm_size') or rd.get('sku_name') or '',
                'region': rd.get('region') or rd.get('location') or '',
            })
    
    results_df = pd.DataFrame(batch_results)
    print(f'Batch processing complete: {len(results_df)} recommendations from {len(resources_pdf)} resources')
    if not results_df.empty:
        print(f'Total estimated annual savings: ${results_df["estimated_savings"].sum():,.2f}')
        print(f'\nBy rule:')
        display(results_df.groupby('rule_code').agg(
            count=('resource_id', 'count'),
            total_savings=('estimated_savings', 'sum'),
        ).sort_values('total_savings', ascending=False))
        print(f'\nTop 10 by savings:')
        display(results_df.nlargest(10, 'estimated_savings'))

## 9. Test GenericPricing (Azure Retail API)

The Retail API works from anywhere — no Glue needed.

In [ ]:
import aiohttp, asyncio

async def test_retail_api_pricing():
    async with aiohttp.ClientSession() as session:
        pricing = GenericPricing(
            config={
                'vmui_base_url': '',
                'vmui_username': '',
                'vmui_password': '',
                'vmui_window_days': 90,
            },
            http_client=session,
            billing_cost_cache=billing_cache,
            sku_catalog_cache=sku_cache,
        )
        
        # Test a VM SKU lookup via Azure Retail API
        test_cases = [
            ('Standard_D4s_v5', 'eastus', 'Virtual Machines'),
            ('Standard_B2s', 'westus2', 'Virtual Machines'),
        ]
        for sku, region, svc in test_cases:
            try:
                cost, source = await pricing.get_annual_cost(
                    resource_id=f'test-{sku}',
                    service_name=svc,
                    region=region,
                    sku_name=sku,
                )
                print(f'  {sku:25s} {region:12s} -> ${cost:>10,.2f}/yr  (source: {source})')
            except Exception as e:
                print(f'  {sku:25s} {region:12s} -> ERROR: {e}')

await test_retail_api_pricing()

## 10. Test with RightSizing Engine Integration

The Azure orchestrator uses `RightSizingEngine` for SKU recommendations.  
Load it the same way as notebook 02.

In [ ]:
from local_helpers import LocalIcebergReader, LocalConfigLoader
from rightsizing_engine.constants import load_mappings, SIZE_ORDINAL, AZURE_TIER_ORDINAL
from rightsizing_engine.sku_catalog import SKUCatalog
from rightsizing_engine.pricing_resolver import PricingResolver
from rightsizing_engine.service_config_loader import ServiceConfigLoader
from rightsizing_engine.sizing_strategies import SizingStrategyFactory
from rightsizing_engine.models import RightsizingRule

CLOUD = 'azure'
reader = LocalIcebergReader()

print('Loading Azure reference data from Athena...')
catalog_pdf = reader.load_sku_catalog(CLOUD)
pricing_pdf = reader.load_pricing_history(CLOUD)
configs_pdf = reader.load_service_configs(CLOUD)
rs_rules_pdf = reader.load_rules(CLOUD)

print(f'  SKU catalog:      {len(catalog_pdf)} rows')
print(f'  Pricing history:  {len(pricing_pdf)} rows')
print(f'  Service configs:  {len(configs_pdf)} rows')
print(f'  RS rules:         {len(rs_rules_pdf)} rows')

# Initialize components
config_loader = LocalConfigLoader()
sizing_defaults = config_loader.load_json('sizing_defaults.json', {})
provider_config = config_loader.load_json('provider_config.json', {})

mappings = load_mappings()
if sizing_defaults:
    mappings['azure_tier_ordinal'] = sizing_defaults.get('azure_tier_ordinal', AZURE_TIER_ORDINAL)
    mappings['size_ordinal'] = sizing_defaults.get('size_ordinal', SIZE_ORDINAL)

provider_config['_sizing_defaults'] = {
    'size_ordinal': mappings.get('size_ordinal', {}),
    'azure_tier_ordinal': mappings.get('azure_tier_ordinal', {}),
}

sku_catalog = SKUCatalog(catalog_pdf, tier_ordinal=mappings.get('azure_tier_ordinal'))
pricing_resolver = PricingResolver(pricing_pdf, sku_catalog, provider_config=provider_config, cloud_provider=CLOUD)

print('\nRightSizing engine components ready for Azure.')

In [ ]:
# Test Azure SKU pricing lookup
test_skus = ['Standard_D4s_v5', 'Standard_D2s_v5', 'Standard_B2s', 'Standard_E4s_v5']

for sku in test_skus:
    price, source = pricing_resolver.get_effective_price(
        'azure', 'Virtual Machines', sku, 'eastus'
    )
    print(f'  {sku:25s} -> ${price:.4f}/hr  (source: {source})')

## 11. Hot-Reload After Code Changes

Edit the source file, then re-run this cell to pick up changes.

In [ ]:
# Re-import the Azure module to pick up code changes
azure_mod = _load_azure_classes()

ThresholdRule           = azure_mod.ThresholdRule
GenericPricing          = azure_mod.GenericPricing
SavingsCalculator       = azure_mod.SavingsCalculator
BaseAzureServiceRecommender = azure_mod.BaseAzureServiceRecommender
AzureRecommendation     = azure_mod.AzureRecommendation

# Re-inject into adapter
db_config.set_threshold_rule_class(ThresholdRule)
# Clear rule cache so reloaded ThresholdRule class is used
db_config.rules_cache = {k: v for k, v in db_config.rules_cache.items() if k.startswith('service_config_')}

print('Azure module reloaded. Caches cleared. Ready for re-testing.')

## 12. Inspect All Available Services

See what services are configured in RDS for Azure.

In [ ]:
from local_helpers import rds_query

rds_query("""
    SELECT service_code, service_name, category, resource_type,
           metrics_table_suffix, resources_table_suffix,
           service_name_filter, is_active
    FROM service_configuration
    WHERE cloud_provider = 'azure'
    ORDER BY priority, service_code
""")